# ANCP T## — D3 Exploratory Data Analysis (EDA)
## IndiGo profitability and market indicators

**Assignment:** D3 EDA | **Course outcomes:** CO2 and CO4 (official descriptions were not included in the supplied workbooks, so no definitions are inferred)  
**Primary business question:** How do quarterly IndiGo profitability margins vary alongside available fuel-price, USD/INR, CPI and WPI indicators, and which variables merit further investigation in a leakage-aware next-quarter PAT-margin model?

**Data supplied:** `CP Master Data.xlsx` and `CP.xlsx`. The master workbook is used as the primary analysis source because it contains a clearly structured financial target sheet and monthly/daily market-indicator sheets. `CP.xlsx` is treated as a secondary/raw-format reference and is audited separately, not silently merged into the primary analysis.

**Scope note:** This is exploratory analysis, not a causal study and not a trained forecasting model. The primary target is quarterly PAT margin; EBITDA margin is treated as a secondary profitability measure. The available primary workbook does not contain passenger counts or load factor, so those relationships cannot be evaluated here.

## 1. Business question and analytical objectives

The analysis will: (1) assess the quality and coverage of the supplied financial and market data; (2) describe the distributions and time patterns of PAT and EBITDA margins and market indicators; (3) examine associations between quarterly profitability and market variables; (4) assess pre-specified hypotheses using suitable exploratory/statistical methods; and (5) translate evidence and limitations into recommendations for later modelling.

### Pre-specified hypotheses (recorded before inspecting analytical results)

These hypotheses are directional research questions, not claims that relationships must exist. Because the sample is small and observational, correlation is not interpreted as causation.

| ID | Hypothesis / rationale | Variables | H₀ / H₁ and planned evidence |
|---|---|---|---|
| H1 | Quarterly oil benchmark price is associated with PAT margin; fuel is a material airline cost proxy. | PAT margin, oil benchmark | H₀: Spearman ρ = 0; H₁: ρ ≠ 0. Spearman correlation, effect size ρ and p-value. |
| H2 | USD/INR is associated with PAT margin; exchange rates may affect USD-linked costs. | PAT margin, USD/INR | H₀: Spearman ρ = 0; H₁: ρ ≠ 0. Spearman correlation and p-value. |
| H3 | CPI inflation is associated with PAT margin. | PAT margin, CPI | H₀: Spearman ρ = 0; H₁: ρ ≠ 0. Spearman correlation and p-value, only if sufficient paired observations. |
| H4 | WPI inflation is associated with PAT margin. | PAT margin, WPI | H₀: Spearman ρ = 0; H₁: ρ ≠ 0. Spearman correlation and p-value, only if sufficient paired observations. |
| H5 | PAT margin differs across fiscal-quarter labels, indicating possible seasonality. | PAT margin, quarter label | H₀: all quarter-group distributions are equal; H₁: at least one differs. Kruskal–Wallis test, group medians and effect-size context; small groups limit power. |
| H6 | PAT and EBITDA margins move together as measures of operating/overall profitability. | PAT margin, EBITDA margin | H₀: Spearman ρ = 0; H₁: ρ ≠ 0. Spearman correlation and p-value. |

**Evidence rules:** report paired sample size, effect size and p-value where appropriate; do not treat a p-value alone as business importance. Hypotheses are not retrofitted after viewing the results. If a variable is unavailable or too sparse, report it as not assessable.

## 2. Imports, configuration, and reproducibility

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")

# Portable configuration: place the source workbooks beside this notebook.
DATA_DIR = Path(".")
MASTER_FILE = DATA_DIR / "CP Master Data.xlsx"
RAW_FILE = DATA_DIR / "CP.xlsx"
# Runtime fallback for the supplied execution environment.
if not MASTER_FILE.exists() and Path("/mnt/data/CP Master Data.xlsx").exists():
    MASTER_FILE = Path("/mnt/data/CP Master Data.xlsx")
if not RAW_FILE.exists() and Path("/mnt/data/CP.xlsx").exists():
    RAW_FILE = Path("/mnt/data/CP.xlsx")

if not MASTER_FILE.exists():
    raise FileNotFoundError("Place 'CP Master Data.xlsx' in the notebook folder.")
print("Master workbook:", MASTER_FILE.resolve())
print("Secondary workbook found:", RAW_FILE.exists())

## 3. Dataset loading and initial inspection

In [ ]:
xls = pd.ExcelFile(MASTER_FILE)
print("Primary workbook sheets:", xls.sheet_names)
raw_sheets = {name: pd.read_excel(MASTER_FILE, sheet_name=name) for name in xls.sheet_names}
for name, frame in raw_sheets.items():
    print(f"\\n{name}: {frame.shape[0]} rows × {frame.shape[1]} columns")
    display(frame.head(3))

## 4. Original-data quality assessment

In [ ]:
def audit_frame(df, name):
    return {
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_cell_pct": float(df.isna().sum().sum() / max(df.size, 1) * 100),
    }

source_audit = pd.DataFrame([audit_frame(df, name) for name, df in raw_sheets.items()])
display(source_audit)

for name, frame in raw_sheets.items():
    print(f"\n{name} — column types and missing values")
    display(pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "missing_n": frame.isna().sum(),
        "missing_pct": (frame.isna().mean()*100).round(2),
        "unique_n": frame.nunique(dropna=True)
    }))

## 5. Original-data representativeness and bias checks

In [ ]:
# The main target sheet is the source population for quarterly target observations.
financial_raw = raw_sheets["Financial source"].copy()
print("Financial target coverage:")
display(pd.DataFrame({
    "metric": ["Rows", "Unique dates", "Duplicate dates", "PAT margin missing", "EBITDA margin missing"],
    "value": [
        len(financial_raw), financial_raw["Date"].nunique(),
        int(financial_raw["Date"].duplicated().sum()),
        int(financial_raw["PAT Margin"].isna().sum()),
        int(financial_raw["EBITDA Margin"].isna().sum())
    ]
}))
print("Important limitation: without a defined target population, sampling design, or external benchmark, statistical representativeness cannot be established.")

## 6. Documented cleaning and transformation

In [ ]:
# Preserve raw sheets above. Cleaned tables are separate copies.
cleaning_log = []

def log_cleaning(dataset, action, reason, impact):
    cleaning_log.append({"dataset": dataset, "action": action, "reason": reason, "potential_impact": impact})

def parse_dates_safely(series, dayfirst=False):
    return pd.to_datetime(series, errors="coerce", dayfirst=dayfirst)

# Financial source
fin = financial_raw.copy()
fin["Date_raw"] = fin["Date"]
fin["Date"] = parse_dates_safely(fin["Date"])
invalid_fin_dates = int(fin["Date"].isna().sum())
if invalid_fin_dates:
    log_cleaning("Financial source", f"Coerced {invalid_fin_dates} invalid dates to missing", "Invalid dates cannot be aligned to quarters", "Rows with invalid dates are excluded from time-series alignment, not silently repaired.")
fin["PAT Margin"] = pd.to_numeric(fin["PAT Margin"], errors="coerce")
fin["EBITDA Margin"] = pd.to_numeric(fin["EBITDA Margin"], errors="coerce")
dupe_fin = int(fin.duplicated(subset=["Date"]).sum())
if dupe_fin:
    log_cleaning("Financial source", f"Found {dupe_fin} duplicate dates", "One target row per quarter is expected", "Duplicates are flagged; no automatic row deletion.")
# No duplicate removal unless verified exact duplicates
exact_dupes = int(fin.drop(columns=["Date_raw"]).duplicated().sum())
if exact_dupes:
    fin = fin.drop_duplicates()
    log_cleaning("Financial source", f"Removed {exact_dupes} exact duplicate rows", "Exact duplicate records add no new information", "Only fully identical rows were removed.")

def clean_market_sheet(df, name, date_col="Date", value_col="PX_LAST", dayfirst=False):
    x = df[[date_col, value_col]].copy()
    x["Date_raw"] = x[date_col]
    x[date_col] = parse_dates_safely(x[date_col], dayfirst=dayfirst)
    x[value_col] = pd.to_numeric(x[value_col], errors="coerce")
    invalid = int(x[date_col].isna().sum())
    if invalid:
        log_cleaning(name, f"Coerced {invalid} invalid dates to missing", "Cannot align malformed dates safely", "Invalid-date observations are excluded from time alignment.")
    missing_val = int(x[value_col].isna().sum())
    if missing_val:
        log_cleaning(name, f"Found {missing_val} missing/non-numeric values", "Numeric indicator required for aggregation", "Missing values remain missing; no imputation in EDA.")
    x = x.dropna(subset=[date_col])
    x = x.sort_values(date_col)
    dupes = int(x.duplicated(subset=[date_col]).sum())
    if dupes:
        log_cleaning(name, f"Found {dupes} duplicate dates", "Repeated dates may affect aggregation", "Kept records; monthly/quarterly aggregation averages available observations.")
    x["quarter_end"] = x[date_col].dt.to_period("Q").dt.end_time.dt.normalize()
    return x

fx = clean_market_sheet(raw_sheets["USD INR"], "USD INR")
cpi = clean_market_sheet(raw_sheets["CPI inflation"], "CPI inflation", dayfirst=True)
wpi = clean_market_sheet(raw_sheets["WPI inflation"], "WPI inflation")
oil_m = clean_market_sheet(raw_sheets["Oil benchmark(MONTHLY)"], "Oil benchmark(MONTHLY)")
oil_d = clean_market_sheet(raw_sheets["Oil benchmark(DAILY)"], "Oil benchmark(DAILY)")

display(pd.DataFrame(cleaning_log))

## 7. Create analysis-ready quarterly panel

In [ ]:
# Quarterly means from market series; financial margins retain source quarter dates.
def quarterly_mean(df, value_col, output_name):
    return (df.dropna(subset=[value_col])
              .groupby("quarter_end", as_index=False)[value_col]
              .mean()
              .rename(columns={value_col: output_name}))

panel = fin.dropna(subset=["Date"]).copy()
panel["quarter_end"] = panel["Date"].dt.to_period("Q").dt.end_time.dt.normalize()
panel = panel.rename(columns={"PAT Margin":"pat_margin_pct", "EBITDA Margin":"ebitda_margin_pct"})
market_quarters = [
    quarterly_mean(fx, "PX_LAST", "usd_inr"),
    quarterly_mean(cpi, "PX_LAST", "cpi_inflation"),
    quarterly_mean(wpi, "PX_LAST", "wpi_inflation"),
    quarterly_mean(oil_m, "PX_LAST", "oil_benchmark")
]
for qdf in market_quarters:
    panel = panel.merge(qdf, on="quarter_end", how="left", validate="many_to_one")
panel = panel.sort_values("quarter_end").reset_index(drop=True)
panel["fiscal_quarter"] = "Q" + panel["quarter_end"].dt.quarter.astype(str)
panel["year"] = panel["quarter_end"].dt.year
panel["quarter_label"] = panel["quarter_end"].dt.to_period("Q").astype(str)

print("Analysis panel shape:", panel.shape)
display(panel.head())
display(panel.tail())

## 8. Cleaned-data quality checks

In [ ]:
clean_audit = pd.DataFrame([
    audit_frame(panel, "Quarterly analysis panel"),
])
display(clean_audit)
display(pd.DataFrame({
    "variable": panel.columns,
    "dtype": [str(panel[c].dtype) for c in panel.columns],
    "missing_n": [int(panel[c].isna().sum()) for c in panel.columns],
    "missing_pct": [(panel[c].isna().mean()*100).round(2) for c in panel.columns],
    "unique_n": [int(panel[c].nunique(dropna=True)) for c in panel.columns]
}))

## 9. Repeated representativeness and bias checks on cleaned data

In [ ]:
def coverage_by_quarter(df, label):
    x = df.copy()
    return pd.DataFrame([{
        "dataset": label,
        "rows": len(x),
        "start_quarter": str(x["quarter_end"].min().date()) if len(x) else None,
        "end_quarter": str(x["quarter_end"].max().date()) if len(x) else None,
        "unique_quarters": x["quarter_end"].nunique(),
        "PAT available": int(x["pat_margin_pct"].notna().sum()) if "pat_margin_pct" in x else np.nan,
        "market_complete_rows": int(x[["usd_inr","cpi_inflation","wpi_inflation","oil_benchmark"]].notna().all(axis=1).sum())
            if all(c in x for c in ["usd_inr","cpi_inflation","wpi_inflation","oil_benchmark"]) else np.nan
    }])

before = pd.DataFrame([{
    "dataset":"Original financial target rows",
    "rows":len(financial_raw),
    "unique_dates":financial_raw["Date"].nunique(),
    "PAT nonmissing":financial_raw["PAT Margin"].notna().sum(),
    "EBITDA nonmissing":financial_raw["EBITDA Margin"].notna().sum()
}])
after = pd.DataFrame([{
    "dataset":"Cleaned quarterly panel",
    "rows":len(panel),
    "unique_dates":panel["quarter_end"].nunique(),
    "PAT nonmissing":panel["pat_margin_pct"].notna().sum(),
    "EBITDA nonmissing":panel["ebitda_margin_pct"].notna().sum()
}])
display(pd.concat([before, after], ignore_index=True))
print("Bias assessment: this is a supplied quarterly company time series, not a random sample of airlines. It is suitable for describing IndiGo over the observed period but cannot establish representativeness for the airline industry.")

## 10. Distribution analysis

In [ ]:
num_cols = ["pat_margin_pct","ebitda_margin_pct","usd_inr","cpi_inflation","wpi_inflation","oil_benchmark"]
available_num = [c for c in num_cols if c in panel.columns]
display(panel[available_num].describe().T)
display(panel[available_num].skew(numeric_only=True).rename("skewness").to_frame())

for col in available_num:
    vals = panel[col].dropna()
    if len(vals) == 0:
        continue
    fig, ax = plt.subplots(figsize=(7, 3.8))
    sns.histplot(vals, kde=(len(vals) >= 5), color="steelblue", ax=ax)
    ax.set_title(f"Distribution of {col.replace('_',' ').title()}")
    ax.set_xlabel(col.replace("_"," ").title())
    ax.set_ylabel("Observation count")
    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(data=panel[available_num], orient="h", color="steelblue", ax=ax)
ax.set_title("Variable distributions and potential extreme values (different units)")
ax.set_xlabel("Value (units differ by variable)")
plt.tight_layout()
plt.show()
print("Box-plot extremes are flags for investigation, not automatic grounds for deletion.")

## 11. Time trends and relationship analysis

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(panel["quarter_end"], panel["pat_margin_pct"], marker="o", label="PAT margin (%)")
ax.plot(panel["quarter_end"], panel["ebitda_margin_pct"], marker="o", label="EBITDA margin (%)")
ax.set_title("Quarterly IndiGo profitability margins")
ax.set_xlabel("Quarter end")
ax.set_ylabel("Margin (%)")
ax.legend()
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

for col in ["usd_inr","cpi_inflation","wpi_inflation","oil_benchmark"]:
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(panel["quarter_end"], panel["pat_margin_pct"], color="tab:blue", marker="o", label="PAT margin (%)")
    ax1.set_xlabel("Quarter end")
    ax1.set_ylabel("PAT margin (%)", color="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot(panel["quarter_end"], panel[col], color="tab:orange", marker=".", label=col)
    ax2.set_ylabel(col.replace("_"," ").title())
    ax1.set_title(f"PAT margin and {col.replace('_',' ').title()} over time (separate scales)")
    plt.xticks(rotation=35)
    plt.tight_layout()
    plt.show()

corr_cols = ["pat_margin_pct","ebitda_margin_pct","usd_inr","cpi_inflation","wpi_inflation","oil_benchmark"]
corr = panel[corr_cols].corr(method="spearman", min_periods=5)
display(corr)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Spearman correlation matrix (pairwise available data)")
plt.tight_layout()
plt.show()

for col in ["usd_inr","cpi_inflation","wpi_inflation","oil_benchmark"]:
    sub = panel[["pat_margin_pct", col]].dropna()
    if len(sub) >= 3:
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.regplot(data=sub, x=col, y="pat_margin_pct", lowess=True,
                    scatter_kws={"alpha":0.8}, line_kws={"color":"black"}, ax=ax)
        ax.set_title(f"PAT margin vs {col.replace('_',' ').title()} (exploratory)")
        ax.set_xlabel(col.replace("_"," ").title())
        ax.set_ylabel("PAT margin (%)")
        plt.tight_layout()
        plt.show()

## 12. Segment analysis

In [ ]:
# Fiscal-quarter segmentation: seasonality is plausible but not assumed.
quarter_summary = (panel.groupby("fiscal_quarter")["pat_margin_pct"]
                   .agg(["count","mean","median","std","min","max"])
                   .reindex(["Q1","Q2","Q3","Q4"]))
display(quarter_summary)
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=panel, x="fiscal_quarter", y="pat_margin_pct",
            order=["Q1","Q2","Q3","Q4"], ax=ax)
sns.stripplot(data=panel, x="fiscal_quarter", y="pat_margin_pct",
              order=["Q1","Q2","Q3","Q4"], color="black", alpha=.55, size=4, ax=ax)
ax.set_title("PAT margin by calendar quarter")
ax.set_xlabel("Calendar quarter")
ax.set_ylabel("PAT margin (%)")
plt.tight_layout()
plt.show()

# Secondary workbook audit: the IS sheet is a wide, multi-header financial export.
if RAW_FILE.exists():
    raw_xls = pd.ExcelFile(RAW_FILE)
    print("Secondary workbook sheets:", raw_xls.sheet_names)
    is_preview = pd.read_excel(RAW_FILE, sheet_name="IS", header=None, nrows=8)
    display(is_preview)
    print("The secondary workbook's IS sheet has multi-row headers and mixed metadata; it is not merged into the clean primary panel in this EDA.")

## 13. Hypothesis assessment and statistical tests

In [ ]:
hypothesis_results = []

def spearman_test(hid, statement, x, y):
    sub = panel[[x,y]].dropna()
    if len(sub) < 5 or sub[x].nunique() < 2 or sub[y].nunique() < 2:
        hypothesis_results.append({"ID":hid,"hypothesis":statement,"n":len(sub),"method":"Spearman correlation",
                                   "effect_size_rho":np.nan,"p_value":np.nan,"assessment":"Not assessable: insufficient paired data or variation"})
        return
    rho, p = stats.spearmanr(sub[x], sub[y])
    hypothesis_results.append({"ID":hid,"hypothesis":statement,"n":len(sub),"method":"Spearman correlation",
                               "effect_size_rho":rho,"p_value":p,
                               "assessment":"Evidence of association at 5% level" if p < .05 else "No statistically significant association detected at 5% level"})

spearman_test("H1","Oil benchmark is associated with PAT margin","oil_benchmark","pat_margin_pct")
spearman_test("H2","USD/INR is associated with PAT margin","usd_inr","pat_margin_pct")
spearman_test("H3","CPI inflation is associated with PAT margin","cpi_inflation","pat_margin_pct")
spearman_test("H4","WPI inflation is associated with PAT margin","wpi_inflation","pat_margin_pct")
spearman_test("H6","PAT and EBITDA margins move together","pat_margin_pct","ebitda_margin_pct")

groups = [g["pat_margin_pct"].dropna().values for _,g in panel.groupby("fiscal_quarter") if g["pat_margin_pct"].notna().sum() >= 2]
n_groups = sum(len(g) for g in groups)
if len(groups) >= 2:
    hstat, hp = stats.kruskal(*groups)
    # Kruskal-Wallis epsilon-squared approximation, clipped to [0,1].
    eps2 = max(0.0, (hstat - len(groups) + 1) / max(n_groups - len(groups), 1))
    hypothesis_results.append({"ID":"H5","hypothesis":"PAT margin differs across calendar quarters","n":n_groups,
                               "method":"Kruskal–Wallis","effect_size_rho":eps2,"p_value":hp,
                               "assessment":"Evidence of group difference at 5% level" if hp < .05 else "No statistically significant group difference detected at 5% level"})
else:
    hypothesis_results.append({"ID":"H5","hypothesis":"PAT margin differs across calendar quarters","n":n_groups,
                               "method":"Kruskal–Wallis","effect_size_rho":np.nan,"p_value":np.nan,
                               "assessment":"Not assessable: insufficient group sizes"})

hypothesis_results = pd.DataFrame(hypothesis_results)
display(hypothesis_results)
print("Interpretation caution: multiple exploratory tests, small time-series sample, autocorrelation and common trends may affect p-values. Findings are associative, not causal.")

## 14. Insight log

In [ ]:
insights = []
def add_insight(iid, insight, evidence, business, modelling, confidence):
    insights.append({"Insight ID":iid,"Insight":insight,"Evidence":evidence,
                     "Business implication":business,"Modelling implication":modelling,
                     "Confidence level":confidence})

# Evidence is generated directly from the supplied data.
if panel["pat_margin_pct"].notna().sum():
    pmin, pmax = panel["pat_margin_pct"].min(), panel["pat_margin_pct"].max()
    pmed = panel["pat_margin_pct"].median()
    add_insight("I01", f"PAT margin varies across the observed period (range {pmin:.2f}% to {pmax:.2f}%; median {pmed:.2f}%).",
                f"n={panel['pat_margin_pct'].notna().sum()}, min={pmin:.2f}%, median={pmed:.2f}%, max={pmax:.2f}%.",
                "A single average may not describe profitability in every quarter.",
                "Use chronological validation and compare a simple baseline against candidate models; inspect extreme quarters.",
                "High for the descriptive range; low for explanations of why it varies.")
for hid in ["H1","H2","H3","H4","H6"]:
    r = hypothesis_results[hypothesis_results["ID"]==hid].iloc[0]
    if pd.notna(r["effect_size_rho"]):
        rho=float(r["effect_size_rho"]); pv=float(r["p_value"])
        add_insight("I"+str(len(insights)+1).zfill(2),
                    f"{r['hypothesis']}: observed Spearman ρ={rho:.2f} (p={pv:.3f}, n={int(r['n'])}).",
                    f"{r['method']}; ρ={rho:.2f}; p={pv:.3f}; n={int(r['n'])}.",
                    "Treat the relationship as exploratory; association does not establish causality.",
                    "Keep the feature as a candidate only; verify value in rolling/expanding time-based validation.",
                    "Medium for association magnitude; low-to-medium for generalisation given small time-series data.")
    else:
        add_insight("I"+str(len(insights)+1).zfill(2),
                    f"{r['hypothesis']}: not assessable with the available paired data.",
                    f"Available paired observations n={int(r['n'])}.",
                    "Do not base a business decision on an untestable relationship.",
                    "Investigate source coverage and avoid imputing values solely to force a test.",
                    "High confidence in the data limitation.")
add_insight("I"+str(len(insights)+1).zfill(2),
            "The supplied dataset is a single-company quarterly time series, not a cross-airline sample.",
            "One company's financial margin observations and aligned market indicators; no population sampling frame supplied.",
            "Conclusions describe IndiGo over this period and may not generalise to other airlines.",
            "Use chronological splits; avoid random train/test splits and report uncertainty due to limited target observations.",
            "High for scope limitation.")
insight_log = pd.DataFrame(insights)
display(insight_log)

## 15. Implications for future modelling

In [ ]:
model_recommendations = pd.DataFrame([
    ["Target", "Use PAT margin (%) as the primary target; keep EBITDA margin as a secondary outcome or benchmark.", "Matches project business question."],
    ["Baseline", "Start with last-quarter PAT margin and/or seasonal-naive (same quarter previous year) baseline.", "Complex models must beat simple benchmarks."],
    ["Candidate models", "Compare regularised linear regression (Ridge/Elastic Net) with one tree-based model only if sample size supports it.", "Quarterly target observations are few; parsimonious models reduce overfitting risk."],
    ["Validation", "Use expanding-window or rolling-origin evaluation; preserve time order.", "Random splits can leak future patterns into training."],
    ["Feature timing", "Use only values available at the forecast origin; aggregate market data to the relevant period using availability cut-offs.", "Prevents look-ahead leakage."],
    ["Market variables", "Keep oil benchmark, USD/INR, CPI and WPI as candidate predictors, subject to missingness and out-of-sample usefulness.", "Exploratory association alone is not proof of predictive value."],
    ["Collinearity", "Inspect correlations among market indicators; consider regularisation and avoid redundant features if unstable.", "Macro series may share trends."],
    ["Outliers", "Investigate extreme quarters against source documents; do not automatically delete them.", "They may reflect genuine shocks or source issues."],
    ["Uncertainty", "Report a forecast interval only after calibrating it on out-of-sample residuals.", "A point forecast alone can imply false precision."],
    ["Deployment", "Log forecast timestamp, input data timestamp, model version, predicted margin, and later actual margin.", "Enables monitoring and retraining decisions."]
], columns=["Area","Recommendation","Evidence-based rationale"])
display(model_recommendations)

## 16. Limitations and uncertainty

- The primary workbook contains 36 quarterly financial observations and market indicators; this is a small effective sample for machine learning.
- CPI includes at least one malformed date in the supplied sheet (the raw date string is retained in the audit trail and the invalid date is not silently repaired). This can leave a quarter without a CPI value.
- The oil series is labelled “Oil benchmark”; it is not explicitly documented as Indian ATF. It must be treated as a proxy, not silently described as ATF.
- Passenger counts, load factor and other airline operating variables are not present in the primary master workbook. They are not invented.
- The data are observational; time trends, autocorrelation, omitted variables, accounting events and macroeconomic shocks can confound associations.
- The secondary `CP.xlsx` has a multi-header financial statement export. It is audited/previewed but not reconciled into the primary dataset because column mapping is ambiguous.
- No sampling frame or population benchmark is supplied, so representativeness cannot be fully established.
- Hypothesis tests are exploratory and multiple comparisons increase false-positive risk.
- No causal claims or predictive performance claims are made by this EDA.

## 17. Conclusions and next steps

In [ ]:
print("FINAL EDA CHECKLIST")
print("Primary workbook:", MASTER_FILE.name)
print("Quarterly target observations:", int(panel["pat_margin_pct"].notna().sum()))
print("Quarterly date span:", panel["quarter_end"].min(), "to", panel["quarter_end"].max())
print("Hypotheses recorded before results:", 6)
print("Hypothesis rows evaluated:", len(hypothesis_results))
print("Insight log rows:", len(insight_log))
print("Cleaning actions logged:", len(cleaning_log))

### Conclusion

This EDA establishes the observed range and time pattern of IndiGo profitability, checks how the supplied macro-market series align to financial quarters, and records exploratory associations without implying causality. The most defensible next step is to build a leakage-aware quarterly master table, verify the exact oil/fuel proxy and financial statement mappings, add documented passenger/load-factor variables if available, and evaluate simple baselines before more complex models. Any eventual production forecast should be validated by historical forecast origin and reported with uncertainty and data-freshness limitations.